Environment Setup & Library Installation

In [1]:
!pip install -q -U transformers accelerate
!pip install -q sentence-transformers
!pip install -q newsapi-python
!pip install -q Pillow requests tqdm
!pip install -q wikipedia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 48.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


IMPORTING LIBRARIES

In [2]:
import transformers, sentence_transformers, torch
import os, re, json, time, warnings
import requests
import numpy as np
import pandas as pd
from io import BytesIO
from PIL import Image
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings('ignore')

Semantic Embedding Module (Sentence Transformers)

In [3]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Dataset Loading & Cleaning

In [5]:
CSV_PATH = "/content/drive/MyDrive/CVProject/dataset_with_clip.csv"

df = pd.read_csv(CSV_PATH)

# Clean dataset
df = df.drop(columns=[c for c in df.columns if 'Unnamed' in c])
df['clean_title'] = df['clean_title'].fillna(df['title'])
df = df.dropna(subset=['clean_title']).reset_index(drop=True)

print("✅ Dataset loaded")
print("Rows:", len(df))
print("Columns:", list(df.columns))

print("\nLabel breakdown (2-way):")
print(df['2_way_label'].value_counts().rename(index={0:'REAL', 1:'FAKE'}))

✅ Dataset loaded
Rows: 26697
Columns: ['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label', '6_way_label', 'clip_score', 'clip_img_feat_path', 'clip_txt_feat_path']

Label breakdown (2-way):
2_way_label
FAKE    14939
REAL    11758
Name: count, dtype: int64


LOAD BLIP MODEL

In [6]:
from transformers import BlipProcessor, BlipForConditionalGeneration
device = "cuda" if torch.cuda.is_available() else "cpu"

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
!pip install -q -U google-generativeai

SET API KEYS

In [ ]:
os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"
os.environ["NEWS_API_KEY"] = "YOUR_API_KEY"

In [57]:
API_KEY = os.getenv("GEMINI_API_KEY")

url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent?key={API_KEY}"

data = {
  "contents": [{
    "parts": [{"text": "Say hello"}]
  }]
}

response = requests.post(url, json=data)

print(response.json())

{'candidates': [{'content': {'parts': [{'text': 'Hello!'}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}], 'usageMetadata': {'promptTokenCount': 2, 'candidatesTokenCount': 2, 'totalTokenCount': 19, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 2}], 'thoughtsTokenCount': 15}, 'modelVersion': 'gemini-2.5-flash', 'responseId': 'kibhadGABNyvqtsP07-g2Qs'}


INIT NEWS API

In [58]:
from newsapi import NewsApiClient
newsapi = NewsApiClient(api_key=os.getenv("NEWS_API_KEY"))

Real-Time News Evidence Retrieval Module

In [59]:
def get_news_evidence(query):
    try:
        trusted_sources = 'bbc-news,the-hindu,cnn'

        articles = newsapi.get_everything(
            q=query,
            sources=trusted_sources,
            language='en',
            page_size=3
        )

        if articles and len(articles['articles']) >= 2:
            selected_articles = articles['articles']
        else:
            fallback_sources = 'reuters,associated-press,al-jazeera-english,independent,dw-news,abc-news'

            fallback = newsapi.get_everything(
                q=query,
                sources=fallback_sources,
                language='en',
                page_size=3
            )

            selected_articles = articles.get('articles', []) + fallback.get('articles', [])

        return " ".join([
            f"{a['title']}. {a.get('description','')}"
            for a in selected_articles
        ])

    except:
        return "No news evidence found"

Knowledge Base Retrieval using Wikipedia

In [60]:
import wikipedia

def get_wikipedia_evidence(query):
    try:
        # Search related pages first
        search_results = wikipedia.search(query)

        if not search_results:
            return "No Wikipedia evidence found"

        # Take top result
        top_page = search_results[0]

        # Get summary
        summary = wikipedia.summary(top_page, sentences=2)

        return summary

    except:
        return "No Wikipedia evidence found"

IMAGE CAPTIONING (BLIP)

In [61]:
def generate_caption(image_url):

    if not image_url or str(image_url) == "nan":
        return "No image description available"

    try:
        response = requests.get(image_url, timeout=5)
        image = Image.open(BytesIO(response.content)).convert("RGB")

        inputs = blip_processor(images=image, return_tensors="pt").to(device)
        output = blip_model.generate(**inputs)

        return blip_processor.decode(output[0], skip_special_tokens=True)

    except:
        return "No image description available"

LLM Reasoning Module

In [62]:
def llm_reasoning(claim, caption, evidence):

    api_key = os.getenv("GEMINI_API_KEY")

    url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash-lite:generateContent?key={api_key}"

    prompt = f"""
You are helping in a fake news detection task.

Claim:
{claim}

Image Description:
{caption}

Evidence:
{evidence}

Instructions:
- If the claim is clearly supported by factual evidence → Real
- If the claim is clearly false or contradicted → Fake
- If the claim is opinion-based, ambiguous, or not fully verifiable → Uncertain

Give output in this format:

Prediction:
Real / Fake / Uncertain

Justification:
Explain briefly

Confidence:
0 to 1
"""

    data = {
        "contents": [
            {
                "parts": [{"text": prompt}]
            }
        ]
    }

    # trying few times in case API fails
    for i in range(3):
        try:
            response = requests.post(url, json=data)
            result = response.json()

            print("Attempt:", i+1)

            # if response is correct
            if "candidates" in result:
                return result.get("candidates", [{}])[0].get("content", {}).get("parts", [{}])[0].get("text", "")

            # if API gives error
            if "error" in result:
                print("API error:", result["error"]["message"])

                # skip if server busy
                if "high demand" in result["error"]["message"].lower():
                    break

                time.sleep(3)

        except Exception as e:
            print("Error:", e)
            time.sleep(3)

    # fallback if nothing works
    return """Prediction:
Uncertain

Justification:
Not enough information or API issue

Confidence:
0.3
"""

LLM Output Parsing

In [63]:

def parse_output(output):

    prediction = "Unknown"
    justification = ""
    confidence = 0.0

    try:
        lines = [line.strip() for line in output.split("\n") if line.strip()]

        for line in lines:

            if line.lower().startswith("prediction"):
                if ":" in line:
                    prediction = line.split(":", 1)[1].strip()

            elif line.lower().startswith("justification"):
                if ":" in line:
                    justification = line.split(":", 1)[1].strip()

            elif line.lower().startswith("confidence"):
                if ":" in line:
                    confidence = float(line.split(":", 1)[1].strip())

        # simple correction
        if prediction.lower() == "uncertain" and confidence > 0.7:
            confidence = 0.5

    except Exception as e:
        print("Parsing error:", e)

    return prediction, justification, confidence

Evidence Ranking [Contextual Similarity Filtering using Sentence Transformers]

In [64]:

def retrieve_evidence(query):

    try:
        news = get_news_evidence(query)
        wiki = get_wikipedia_evidence(query)

        docs = []

        if wiki and wiki != "No Wikipedia evidence found":
            docs.append(wiki)

        if news and news != "No news evidence found":
            docs.extend(news.split(". "))

        if len(docs) == 0:
            return f"No external evidence found. Analyze based on claim: {query}"

        query_emb = embedder.encode(query, convert_to_tensor=True)
        doc_embs = embedder.encode(docs, convert_to_tensor=True)

        scores = util.cos_sim(query_emb, doc_embs)[0]

        top_k = min(3, len(docs))
        top_indices = scores.argsort(descending=True)[:top_k]

        top_docs = [docs[i] for i in top_indices]

        return " ".join(top_docs)

    except Exception as e:
        print("Retrieval error:", e)
        return "No external evidence found"

Main Pipeline Function

In [65]:
def run_pipeline(row):

    claim = row['title']
    image = row['image_url']

    # Step 1: Caption
    caption = generate_caption(image)

    # Step 2: Evidence
    evidence = retrieve_evidence(claim)

    # Step 3: LLM
    llm_output = llm_reasoning(claim, caption, evidence)

    # Step 4: Parse
    prediction, justification, confidence = parse_output(llm_output)

    return {
        "Claim": claim,
        "Caption": caption,
        "Evidence": evidence,
        "Prediction": prediction,
        "Justification": justification,
        "Confidence": confidence
    }

TEST RUN

In [18]:
result = run_pipeline(df.iloc[0])
print(result)

Attempt: 1
{'Claim': 'pd: phoenix car thief gets instructions from youtube video', 'Caption': 'two men are seen in this und photo', 'Evidence': 'No external evidence found. Analyze based on claim: pd: phoenix car thief gets instructions from youtube video', 'Prediction': 'Uncertain', 'Justification': 'The claim is about a specific event involving a "phoenix car thief" and instructions from a YouTube video. The provided image description is very general ("two men are seen in this und photo") and offers no details related to the claim. Since there is no external evidence to corroborate or refute the claim, it remains unverified and therefore uncertain.', 'Confidence': 0.2}


In [38]:
result = run_pipeline(df.iloc[1])
print(result)

Attempt: 1
Parsing error: could not convert string to float: ''
{'Claim': 'as trump accuses iran, he has one problem: his own credibility', 'Caption': 'president donald trump speaks at a rally in washington', 'Evidence': 'During and between his terms as President of the United States, Donald Trump has made tens of thousands of false or misleading claims. Fact-checkers at The Washington Post documented 30,573 false or misleading claims during his first presidential term, an average of 21 per day.', 'Prediction': '', 'Justification': '', 'Confidence': 0.0}


In [39]:
result = run_pipeline(df.iloc[11])
print(result)

Attempt: 1
{'Claim': 'bird poop that looked like a bird', 'Caption': 'a car is stopped at a red light', 'Evidence': 'Seabirds (also known as marine birds) are birds that are adapted to life within the marine environment. While seabirds vary greatly in lifestyle, behaviour and physiology, they often exhibit striking convergent evolution, as the same environmental problems and feeding niches have resulted in similar adaptations.', 'Prediction': 'Fake', 'Justification': 'The evidence provided discusses seabirds and their adaptations, which is completely irrelevant to the claim about bird poop resembling a bird. The image description of a car at a red light also offers no support. There is no factual basis to suggest bird poop would look like a bird.', 'Confidence': 1.0}


In [40]:
result = run_pipeline(df.iloc[54])
print(result)

Attempt: 1
{'Claim': 'saw a fellow riding a unicycle when i was walking home from work.', 'Caption': 'a street with a red car parked on the side', 'Evidence': 'This is a list of the episodes of Mad, an animated sketch comedy television series inspired by Mad magazine that aired on Cartoon Network.', 'Prediction': 'Uncertain', 'Justification': 'The claim is about a personal observation of someone riding a unicycle. The provided image description of a street with a parked red car and the evidence about the animated series "Mad" are completely irrelevant to the claim. There is no factual evidence to support or refute the claim.', 'Confidence': 0.1}


In [ ]:
TEST_ROWS = 100
TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/100test_run_output.csv"

from datetime import datetime

print("🧪 Running verification on 10 rows...")

# take only first 10 rows
test_df = df.head(TEST_ROWS).copy()

# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# 🔁 LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        # store outputs
        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE TEST OUTPUT
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("✅ Test run completed!")
print(f"📂 File saved at: {TEST_SAVE_PATH}")

🧪 Running verification on 10 rows...


  0%|          | 0/100 [00:00<?, ?it/s]

Attempt: 1


  1%|          | 1/100 [00:06<10:27,  6.34s/it]

Attempt: 1
Parsing error: could not convert string to float: ''


...

Attempt: 1
API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 39.29134868s.
Attempt: 2
API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 36.236445163s.
Attempt: 3
API error: You exceeded your current quota, please check your plan and billing details. For more information on this error

 99%|█████████▉| 99/100 [22:48<00:15, 15.06s/it]

Attempt: 1
API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 24.597900965s.
Attempt: 2
API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 21.517106474s.
Attempt: 3
API error: You exceeded your current quota, please check your plan and billing details. For more information on this erro

100%|██████████| 100/100 [23:03<00:00, 13.83s/it]

✅ Test run completed!
📂 File saved at: /content/drive/MyDrive/CVProject/100test_run_output.csv


In [ ]:
START_ROW = 17   # 👈 change this (last processed row)
END_ROW = 40

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/test_run_output2.csv"

from datetime import datetime

print(f"🧪 Running verification from row {START_ROW} to {END_ROW}...")

# ✅ slice dataset
test_df = df.iloc[START_ROW:END_ROW].copy().reset_index(drop=True)

# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# 🔁 LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("✅ Test run completed!")
print(f"📂 File saved at: {TEST_SAVE_PATH}")

🧪 Running verification from row 17 to 40...


  0%|          | 0/23 [00:00<?, ?it/s]

Attempt: 1


  4%|▍         | 1/23 [00:09<03:30,  9.59s/it]

Attempt: 1


...

Attempt: 1
API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 6.688388901s.
Attempt: 2


 96%|█████████▌| 22/23 [03:08<00:12, 12.32s/it]

Attempt: 1
API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 54.957214145s.
Attempt: 2


100%|██████████| 23/23 [03:18<00:00,  8.63s/it]

✅ Test run completed!
📂 File saved at: /content/drive/MyDrive/CVProject/test_run_output2.csv


In [ ]:
MISSING_ROW = 40

test_df = df.iloc[MISSING_ROW:MISSING_ROW+1].copy().reset_index(drop=True)

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/missing_test_run_output40.csv"

from datetime import datetime

print(f"🧪 Running verification from row {START_ROW} to {END_ROW}...")



# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# 🔁 LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("✅ Test run completed!")
print(f"📂 File saved at: {TEST_SAVE_PATH}")

🧪 Running verification from row 41 to 60...


  0%|          | 0/1 [00:00<?, ?it/s]

Attempt: 1


100%|██████████| 1/1 [00:09<00:00,  9.12s/it]

✅ Test run completed!
📂 File saved at: /content/drive/MyDrive/CVProject/missing_test_run_output40.csv


In [ ]:
START_ROW = 61   # 👈 change this (last processed row)
END_ROW = 80

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/test_run_output4.csv"

from datetime import datetime

print(f"🧪 Running verification from row {START_ROW} to {END_ROW}...")

# ✅ slice dataset
test_df = df.iloc[START_ROW:END_ROW].copy().reset_index(drop=True)

# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# 🔁 LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("✅ Test run completed!")
print(f"📂 File saved at: {TEST_SAVE_PATH}")

🧪 Running verification from row 61 to 80...


  0%|          | 0/19 [00:00<?, ?it/s]

Attempt: 1


  5%|▌         | 1/19 [00:08<02:37,  8.77s/it]

Attempt: 1
Parsing error: could not convert string to float: ''


...

Attempt: 1
Parsing error: could not convert string to float: ''


 95%|█████████▍| 18/19 [02:15<00:08,  8.76s/it]

Attempt: 1


100%|██████████| 19/19 [02:22<00:00,  7.50s/it]

✅ Test run completed!
📂 File saved at: /content/drive/MyDrive/CVProject/test_run_output4.csv


In [ ]:
MISSING_ROW = 60

test_df = df.iloc[MISSING_ROW:MISSING_ROW+1].copy().reset_index(drop=True)

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/missing_test_run_output60.csv"

from datetime import datetime

print(f"🧪 Running verification from row {START_ROW} to {END_ROW}...")



# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# 🔁 LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("✅ Test run completed!")
print(f"📂 File saved at: {TEST_SAVE_PATH}")

🧪 Running verification from row 61 to 80...


  0%|          | 0/1 [00:00<?, ?it/s]

Attempt: 1


100%|██████████| 1/1 [00:08<00:00,  8.43s/it]

✅ Test run completed!
📂 File saved at: /content/drive/MyDrive/CVProject/missing_test_run_output60.csv


In [ ]:
START_ROW = 80   # 👈 change this (last processed row)
END_ROW = 100

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/test_run_output5.csv"

from datetime import datetime

print(f"🧪 Running verification from row {START_ROW} to {END_ROW}...")

# ✅ slice dataset
test_df = df.iloc[START_ROW:END_ROW].copy().reset_index(drop=True)

# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# 🔁 LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("✅ Test run completed!")
print(f"📂 File saved at: {TEST_SAVE_PATH}")

🧪 Running verification from row 80 to 100...


  0%|          | 0/20 [00:00<?, ?it/s]

Attempt: 1


  5%|▌         | 1/20 [00:08<02:49,  8.93s/it]

Attempt: 1


...

Attempt: 1


 95%|█████████▌| 19/20 [02:19<00:07,  7.22s/it]

Attempt: 1


100%|██████████| 20/20 [02:26<00:00,  7.33s/it]

✅ Test run completed!
📂 File saved at: /content/drive/MyDrive/CVProject/test_run_output5.csv


In [68]:
import os

files = os.listdir("/content/drive/MyDrive/CVProject")

for f in files:
    if "test_run" in f:
        print(f)

test_run_output1.csv
test_run_output2.csv
test_run_output3.csv
missing_test_run_output40.csv
test_run_output4.csv
missing_test_run_output60.csv
test_run_output5.csv


In [ ]:
import pandas as pd

# ✅ Your files in correct order
files = [
    "/content/drive/MyDrive/CVProject/test_run_output1.csv",
    "/content/drive/MyDrive/CVProject/test_run_output2.csv",
    "/content/drive/MyDrive/CVProject/missing_test_run_output40.csv",
    "/content/drive/MyDrive/CVProject/test_run_output3.csv",
    "/content/drive/MyDrive/CVProject/missing_test_run_output60.csv",
    "/content/drive/MyDrive/CVProject/test_run_output4.csv",
    "/content/drive/MyDrive/CVProject/test_run_output5.csv"
]

# 🔁 Read and store all
dfs = []
for file in files:
    print(f"📂 Loading: {file}")
    df = pd.read_csv(file)
    dfs.append(df)

# 🔗 Merge in same order
merged_df = pd.concat(dfs, ignore_index=True)

# Save final file
SAVE_PATH = "/content/drive/MyDrive/CVProject/final_llm_score_dataset.csv"
merged_df.to_csv(SAVE_PATH, index=False)

print("✅ All files merged successfully!")
print("📂 Saved at:", SAVE_PATH)

📂 Loading: /content/drive/MyDrive/CVProject/test_run_output1.csv
📂 Loading: /content/drive/MyDrive/CVProject/test_run_output2.csv
📂 Loading: /content/drive/MyDrive/CVProject/missing_test_run_output40.csv
📂 Loading: /content/drive/MyDrive/CVProject/test_run_output3.csv
📂 Loading: /content/drive/MyDrive/CVProject/missing_test_run_output60.csv
📂 Loading: /content/drive/MyDrive/CVProject/test_run_output4.csv
📂 Loading: /content/drive/MyDrive/CVProject/test_run_output5.csv
✅ All files merged successfully!
📂 Saved at: /content/drive/MyDrive/CVProject/final_llm_score_dataset.csv
